# COM-490: Robust Public Transport Routing Engine
**Group X1** | Final Project

By: Matthias Wyss, Thierry Sokhn, Léan Bruttin, Alain Girard and Sofia Taouhid

This notebook demonstrates a robust journey planner for the SBB (Swiss Federal Railways) network. Unlike traditional planners that strictly minimize travel time, our engine guarantees an arrival time with a defined statistical confidence level ($Q\%$) by modeling the probability of connection failures using historical data.

## 1. Environment Setup & Authentication
This section initializes the environment, retrieves the secure JWT tokens from the EPFL COM-490 variables, and establishes the user namespaces for distributed data storage on HDFS.

In [1]:
# Automatically reload imported custom modules if modified on disk
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import json
import random
import re
import datetime
import warnings
import base64 as b64
from random import randrange
from urllib.parse import urlparse
from contextlib import closing

import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, Window
from trino.dbapi import connect
from trino.auth import JWTAuthentication

# Custom project modules
from DelayModel import build_delay_model
from JourneyPlanner import JourneyPlanner, prepare_graph, DAY_ORDER
from visualization import show_dashboard

# Suppress specific non-critical SQLAlchemy warnings for cleaner presentation output
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.filterwarnings("ignore", category=UserWarning, message="pandas only supports SQLAlchemy connectable .*")

In [2]:
groupName = 'X1'
hadoopfs = os.environ.get('HADOOP_FS')

def getUsername():
    """
    Safely extracts the username and token validity from the JWT (JSON Web Token) 
    provided by the EPFL COM-490 environment variables.
    """
    # Extract the payload segment of the JWT
    payload = os.environ.get('EPFL_COM490_TOKEN').split('.')[1]
    # Add necessary padding for base64 decoding
    payload = payload + '=' * (4 - len(payload) % 4)
    obj = json.loads(b64.urlsafe_b64decode(payload))
    
    # Check if the token is close to expiration (1 hour buffer)
    if (time.time() > int(obj.get('exp')) - 3600):
        raise Exception('Your credentials have expired, please restart your Jupyter Hub server:'
                        'File>Hub Control Panel, Stop My Server, Start My Server.')
    
    time_left = int((obj.get('exp') - time.time())/3600)
    return obj.get('sub'), time_left

# Initialize user context variables
username, validity_h = getUsername()
hadoopfs = os.environ.get('HADOOP_FS')

# Define database namespaces (schemas)
sharedns = 'iceberg.com490_iceberg'           # Read-only shared course data
userns = 'iceberg.' + username + '_iceberg'   # Personal writable workspace
groupfs  = f"{hadoopfs}/user/groups/com-490/{groupName}"

# Validate group naming convention
if not re.search('([A-Z][0-9Z])', groupName):
    raise Exception(f"Invalid group name {groupName}")

print(f"You are: {username} of group {groupName}")
print(f"credentials validity: {validity_h} hours left.")
print(f"personal namespace:   {userns}")
print(f"Group HDFS folder:    {groupfs}")

You are: agirard of group X1
credentials validity: 167 hours left.
personal namespace:   iceberg.agirard_iceberg
Group HDFS folder:    hdfs://iccluster061.iccluster.epfl.ch:9000/user/groups/com-490/X1


## 2. Distributed Computing Initialization
To process the massive volume of SBB historical data and GTFS timetables, we rely on Trino and Spark.

In [3]:
# Authenticate against the Trino coordinator using the JWT token
trinoAuth = JWTAuthentication(os.environ.get('EPFL_COM490_TOKEN'))
trinoUrl  = urlparse(os.environ.get('TRINO_URL'))

# Establish a connection to Trino to execute fast SQL queries on Iceberg tables
conn = connect(
    host=trinoUrl.hostname,
    port=trinoUrl.port,
    auth=trinoAuth,
    http_scheme=trinoUrl.scheme,
    verify=True
)
print('Trino connected!')

Trino connected!


In [4]:
# Initialize a distributed Spark Session on the YARN resource manager
spark = (SparkSession\
            .builder
            .appName(username + '-final-project')
            # Assign a random UI port to avoid collisions with other students on the cluster
            .config('spark.ui.port', randrange(4050, 4450, 5))
            .config("spark.executorEnv.PYTHONPATH", ":".join(sys.path))
            
            # Inject required dependencies for Iceberg and Sedona (Geospatial) catalogs
            .config('spark.jars',
                    f'{hadoopfs}/data/com-490/jars/iceberg-spark-runtime-3.5_2.13-1.6.1.jar,'
                    f'{hadoopfs}/data/com-490/jars/sedona-spark-shaded-3.5_2.13-1.7.1.jar,'
                    f'{hadoopfs}/data/com-490/jars/geotools-wrapper-1.7.1-28.5.jar'
            )
            
            # Configure Apache Iceberg as the primary table format provider
            .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
            .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
            .config('spark.sql.catalog.iceberg.type', 'hadoop')
            .config('spark.sql.catalog.iceberg.warehouse', f'{hadoopfs}/data/com-490/silver/')
            
            # Configure personal workspace catalog for saving intermediate parquet files
            .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
            .config('spark.sql.catalog.spark_catalog.type', 'hadoop')
            .config('spark.sql.catalog.spark_catalog.warehouse', f'{hadoopfs}/user/{username}/final-project/warehouse')
            .config("spark.sql.warehouse.dir", f'{hadoopfs}/user/{username}/final-project/spark/warehouse')
            
            # Allocate cluster resources (4 executors with 4 cores and 6GB RAM each)
            .config("spark.executor.memory", "6g")
            .config("spark.executor.cores", "4")
            .config("spark.executor.instances", "4")
        ).master('yarn').getOrCreate()

print('Spark connected!')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 11:34:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Spark connected!


In [5]:
spark.sparkContext

<SparkContext master=yarn appName=agirard-final-project>

## 3. Spatial & Temporal Graph Construction
Before routing, we extract a constrained subset of the Swiss public transport network. Based on the project specifications, we filter the graph to keep only Lausanne and Ouest Lausannois districts, restrict pedestrian transfers to a maximum of 500 meters, and assume a standard walking speed of $50\text{ m/min}$.

In [6]:
# Target administrative regions for the routing planner
uuid_lausanne = "a7a21b73-6ffe-4fbf-a635-6e2b961f3072"
uuid_ouest_lausannois = "e168fd57-f57a-4075-a350-0dcfbb55147f"
uuids = [uuid_lausanne, uuid_ouest_lausannois]

# Physical routing constraints (derived from project specifications)
MAX_WALK_DISTANCE_M = 500       # Maximum allowed pedestrian distance between stops
MIN_TRANSFER_TIME_SEC = 120     # Base penalty (2 mins) required to change platforms
WALKING_SPEED_M_PER_MIN = 50    # Assumed pedestrian speed (50 meters per minute)

In [7]:
# Set to True for the first run or if region UUIDs/walking constraints are modified.
# When False, drastically speeds up execution by loading pre-computed tables from Trino.
REBUILD_GRAPH = True

# Extract stops, pedestrian transfers, and schedule metadata
stops_df, transfers_df, stop_times_df = prepare_graph(
    conn=conn,
    sharedns=sharedns,
    userns=userns,
    uuids=uuids,
    max_walk_m=MAX_WALK_DISTANCE_M,
    rebuild=REBUILD_GRAPH,
    verbose=True,
)

Preparing topological graph tables in Trino...
Reference timetable publication date: 2026-01-31
Topological graph loaded: 390 stops, 1247 walking edges, 980693 scheduled stops (33.1s).


## 4. Baseline Routing & Algorithmic Framework

With the spatial and temporal network topology loaded, the core routing engine solves the **Latest Departure Problem**. Given a strict arrival deadline ($T_{target}$) and a minimum statistical confidence threshold ($Q\%$), the system implements a **Backward Stochastic Multi-Criteria Time-Dependent Dijkstra** algorithm. 

### Backward Search Logic
Unlike conventional planners that search forward from a start time, our engine traverses the network backwards from the destination. By initiating the search at the target arrival time ($T_{target}$) and subtracting edge weights (transit times, transfer penalties, and walking durations), the algorithm naturally discovers the latest possible departure time that still satisfies all arrival constraints.

### Multi-Criteria Pareto Dominance Frontier
In a complex transit network, optimizing for a single metric (e.g., travel time) often neglects user comfort and reliability. Our algorithm maintains a **Pareto Dominance Frontier** at each station node, tracking three conflicting objectives simultaneously:
1. **Departure Time** (to be maximized backwards)
2. **Path Confidence** (the product of connection probabilities, $P(success) \ge Q\%$)
3. **Cumulative Walking Distance** (to be minimized)

A route state is considered "Pareto Optimal" if no other known route provides a later departure, higher confidence, and shorter walking distance simultaneously. 

### The Search Process
During the expansion phase, the algorithm explores two types of edges:
* **Pedestrian Edges:** Transfers between stops are evaluated based on their physical distance, with an added temporal penalty based on the SBB transfer formula (base transfer time + walking time).
* **Transit Edges (Time-Dependent):** The algorithm "boards" a vehicle only if the scheduled arrival at the next stop allows for a successful transfer (accounting for potential delays via our `DelayModel`). 

If a branch's cumulative confidence $Q$ drops below the threshold, or if a state is strictly dominated by an existing Pareto-optimal label, the branch is pruned to optimize search performance. This ensures that the engine only presents the user with non-redundant, high-quality itineraries.

### The Hierarchical Fallback Delay Model
To estimate connection reliability and mitigate data sparsity issues, we deploy an empirical delay model that dynamically cascades through a 5-level hierarchical fallback structure. The system always attempts to retrieve the most granular data available, falling back to broader aggregates when observation counts drop below a significance threshold ($\ge 20$):

* **Level 1 (trip, stop, weather):** Specific historical delay distribution for a `trip_id` at a `bpuic` under specific environmental conditions (e.g., adverse weather).
* **Level 2 (trip, stop):** Historical delay distribution for a specific `trip_id` at a specific `bpuic` (weather-agnostic).
* **Level 3 (stop, weather):** Station-wide delay distribution pooling all transit vehicles crossing a specific `bpuic` under specific environmental conditions.
* **Level 4 (stop):** Station-wide delay distribution pooling all transit vehicles crossing a specific `bpuic` (weather-agnostic).
* **Level 5 (network-wide):** Network-wide delay distribution acting as an absolute statistical safety net.

*Note: In the current baseline configuration (Part 1), the model operates in a standard mode, utilizing only Levels 2, 4, and 5. Upon activating the weather-aware module (Part 2), the model dynamically expands its lookup logic to fully leverage the contextual precision of Levels 1 and 3.*

In [8]:
# ==============================================================================
# STANDARD PLANNER INITIALIZATION
# ==============================================================================
print("Compiling baseline empirical delay distributions...")

# Ensure heavy Spark computations only run once per kernel session
if "PART1_DONE" not in globals():
    run_align = True
    run_delays = True
    PART1_DONE = True
else:
    run_align = False
    run_delays = False

# Build the baseline statistical delay model
delay_outputs_std = build_delay_model(
    spark=spark,
    hadoopfs=hadoopfs,
    username=username,
    stops_df=stops_df,
    recompute_alignment=run_align,
    recompute_delays=run_delays,
    use_weather_features=False,   # Standard operational mode
    min_obs_for_ccdf=20,          # Minimum sample threshold for statistical trust
    diagnostics=False,
    return_intermediates=False,
    verbose=True,
)

delay_model_std = delay_outputs_std["delay_model"]

# Instantiate the routing engine with baseline constraints
planner_std = JourneyPlanner(
    delay_model=delay_model_std,
    max_walk_distance_m=MAX_WALK_DISTANCE_M,
    min_transfer_time_sec=MIN_TRANSFER_TIME_SEC, 
    walking_speed_m_per_min=WALKING_SPEED_M_PER_MIN
)

# Hydrate the internal search index
planner_std.prepare_from_dataframes(stops_df, transfers_df, stop_times_df)
print("\nStandard Planner is ready for routing.")

Compiling baseline empirical delay distributions...
Computing Istdaten/timetable alignment...


Saving alignment mapping to: hdfs://iccluster061.iccluster.epfl.ch:9000/user/agirard/final-project/aligned_trips.parquet


26/05/27 11:35:06 WARN DAGScheduler: Broadcasting large task binary with size 1048.4 KiB
26/05/27 11:35:28 WARN DAGScheduler: Broadcasting large task binary with size 1146.1 KiB
26/05/27 11:35:31 WARN DAGScheduler: Broadcasting large task binary with size 1167.9 KiB
26/05/27 11:35:56 WARN DAGScheduler: Broadcasting large task binary with size 1169.6 KiB
26/05/27 11:36:11 WARN DAGScheduler: Broadcasting large task binary with size 1384.3 KiB


Alignment execution ready (89.9s).
Computing raw delays from source observations...


Raw delays saved to hdfs://iccluster061.iccluster.epfl.ch:9000/user/agirard/final-project/raw_delays.parquet (49.0s).


DelayModel initialization pipeline complete:
  min_obs: 20
  is_weather_aware: 0
  level1_keys: 0
  level2_keys: 176913
  level3_keys: 0
  level4_keys: 360
  global_observations: 13053539
Search engine index initialized: 390 stops, 1247 walking edges, 55828 unique trips.

Standard Planner is ready for routing.


In [9]:
print("Executing backward routing search...")

# Query the standard planner
routes_std = planner_std.route(
    start_id=8501118,  # Origin: Renens VD
    end_id=8501181,    # Destination: Lausanne-Flon
    arrival_time="08:30:00",
    day="Monday",
    min_confidence=0.90,
    weather=None,      # No specific weather context injected
    max_routes=5,
)

# Display the reconstructed itineraries
JourneyPlanner.print_routes(routes_std)

# Output telemetry to analyze data sparsity handling
print("\nBaseline Model Telemetry (Fallback Layer Distribution):")
print(planner_std.delay_model.usage_stats())

Executing backward routing search...
Found 5 route(s).

=== ROUTE 1 ===
Latest departure: 08:08:00
Arrival time: 08:25:00
Arrival deadline: 08:30:00
Confidence: 93.8%
Walking distance: 438.5 m
  [08:08:00] Board Trip 275.TA.91-1-D-j26-1.16.H at Stop 8501118
  [08:14:00] Arrive at Stop 8501120 via Trip 275.TA.91-1-D-j26-1.16.H
  [08:19:00] Walk from Stop 8501120 to Stop 8501181

=== ROUTE 2 ===
Latest departure: 08:04:00
Arrival time: 08:21:00
Arrival deadline: 08:30:00
Confidence: 97.6%
Walking distance: 438.5 m
  [08:04:00] Board Trip 273.TA.91-95-j26-1.23.H at Stop 8501118
  [08:10:00] Arrive at Stop 8501120 via Trip 273.TA.91-95-j26-1.23.H
  [08:19:00] Walk from Stop 8501120 to Stop 8501181

=== ROUTE 3 ===
Latest departure: 08:04:00
Arrival time: 08:25:00
Arrival deadline: 08:30:00
Confidence: 93.5%
Walking distance: 144.5 m
  [08:04:00] Board Trip 273.TA.91-95-j26-1.23.H at Stop 8501118
  [08:10:00] Arrive at Stop 8501120 via Trip 273.TA.91-95-j26-1.23.H
  [08:16:00] Walk from Sto

### 4.1 Telemetry Analysis: Standard Framework
During the pathfinding execution, the routing algorithm explored thousands of potential connections. The telemetry output above illustrates exactly how our hierarchical delay model manages data sparsity on a standard request:

* **Level 2 (trip, stop):** In over a third of the evaluations, the engine found robust historical data ($\ge 20$ observations) for the exact trip at the exact stop. This represents our highest confidence baseline data.
* **Level 4 (stop):** The majority of the explored connections triggered this fallback. SBB frequently generates new `trip_id` strings for minor timetable updates, meaning many scheduled trips lack deep historical logs. Instead of failing, the model intelligently falls back to the station's average delay distribution, preserving geographical accuracy.
* **Level 5 (network-wide):** A tiny fraction of connections lacked even station-level data, safely defaulting to the overall network distribution to prevent algorithmic crashes.

## 5. Weather-Aware Routing

While the baseline model provides robust results under nominal conditions, adverse meteorological events (such as heavy precipitation) significantly distort transit delay distributions, increasing variance and extending the right-tail risk of missed connections. 

To model this environmental risk, we expand our empirical delay model from a 3-level fallback to our full 5-level operational hierarchy. We introduce weather stratification buckets (`normal` vs. `adverse`), determined by a localized hourly precipitation threshold ($\ge 0.5\text{ mm}$), allowing the engine to distinguish between delay profiles during clear skies and severe storm events.

In [10]:
# ==============================================================================
# WEATHER-AWARE PLANNER INITIALIZATION
# ==============================================================================
print("Building weather-stratified empirical delay model...")

# Dynamic state flag for weather-specific raw delay compilation
if "PART2_DONE" not in globals():
    run_wx_delays = True
    PART2_DONE = True
else:
    run_wx_delays = False

delay_outputs_wx = build_delay_model(
    spark=spark,
    hadoopfs=hadoopfs,
    username=username,
    stops_df=stops_df,
    aligned_trips_df=delay_outputs_std["aligned_trips_df"], 
    recompute_alignment=False, 
    
    recompute_delays=run_wx_delays,
    use_weather_features=True,    # Activate structural weather stratification
    weather_site="LSGL",          # Reference station (Lausanne Airport)
    precip_threshold=0.5,         # Threshold (mm/h) to classify weather as 'adverse'
    min_obs_for_ccdf=20,
    diagnostics=False,
    return_intermediates=False,
    verbose=True,
)

delay_model_wx = delay_outputs_wx["delay_model"]

planner_wx = JourneyPlanner(
    delay_model=delay_model_wx,
    max_walk_distance_m=MAX_WALK_DISTANCE_M,
    min_transfer_time_sec=MIN_TRANSFER_TIME_SEC, 
    walking_speed_m_per_min=WALKING_SPEED_M_PER_MIN
)

planner_wx.prepare_from_dataframes(stops_df, transfers_df, stop_times_df)
print("\nWeather-Aware Planner is ready for routing.")

Building weather-stratified empirical delay model...
Computing raw delays from source observations...


Raw delays saved to hdfs://iccluster061.iccluster.epfl.ch:9000/user/agirard/final-project/raw_delays_weather.parquet (47.0s).


Building contextual weather stratification matrix layers...


26/05/27 11:38:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DelayModel initialization pipeline complete:
  min_obs: 20
  is_weather_aware: 1
  level1_keys: 184221
  level2_keys: 176913
  level3_keys: 711
  level4_keys: 360
  global_observations: 13053539
Search engine index initialized: 390 stops, 1247 walking edges, 55828 unique trips.

Weather-Aware Planner is ready for routing.


In [11]:
print("Executing weather-aware backward routing search...")

# Rerun the exact same query, but inject the environmental constraint
routes_wx = planner_wx.route(
    start_id=8501118,
    end_id=8501181,
    arrival_time="08:30:00",
    day="Monday",
    min_confidence=0.90,
    weather="adverse", # <--- Contextual weather injection
    max_routes=5,
)

print("\n=== PROPOSED ROUTES UNDER ADVERSE WEATHER ===")
JourneyPlanner.print_routes(routes_wx)

# Analyze the telemetry shift to prove context-awareness
print("\nWeather Model Telemetry (Fallback Layer Distribution):")
print(planner_wx.delay_model.usage_stats())

Executing weather-aware backward routing search...

=== PROPOSED ROUTES UNDER ADVERSE WEATHER ===
Found 5 route(s).

=== ROUTE 1 ===
Latest departure: 08:08:00
Arrival time: 08:25:00
Arrival deadline: 08:30:00
Confidence: 96.3%
Walking distance: 438.5 m
  [08:08:00] Board Trip 275.TA.91-1-D-j26-1.16.H at Stop 8501118
  [08:14:00] Arrive at Stop 8501120 via Trip 275.TA.91-1-D-j26-1.16.H
  [08:19:00] Walk from Stop 8501120 to Stop 8501181

=== ROUTE 2 ===
Latest departure: 08:04:00
Arrival time: 08:21:00
Arrival deadline: 08:30:00
Confidence: 98.2%
Walking distance: 438.5 m
  [08:04:00] Board Trip 273.TA.91-95-j26-1.23.H at Stop 8501118
  [08:10:00] Arrive at Stop 8501120 via Trip 273.TA.91-95-j26-1.23.H
  [08:19:00] Walk from Stop 8501120 to Stop 8501181

=== ROUTE 3 ===
Latest departure: 08:04:00
Arrival time: 08:25:00
Arrival deadline: 08:30:00
Confidence: 95.1%
Walking distance: 144.5 m
  [08:04:00] Board Trip 273.TA.91-95-j26-1.23.H at Stop 8501118
  [08:10:00] Arrive at Stop 850112

### 5.1 Telemetry Analysis: Weather-aware Framework
By executing the exact same path exploration under `weather="adverse"`, the telemetry profile shifts dynamically. This proves the context-awareness of the model:

* **Level 1 (trip, stop, weather):** The "Holy Grail" of our data. The engine successfully found sufficient historical observations of the exact train, at the exact stop, specifically under rainy conditions.
* **Level 2 (trip, stop):** Some trains have great historical data overall, but not enough specifically under rain. The model preserves statistical integrity by falling back to the train's standard, weather-agnostic distribution.
* **Level 3 (stop, weather):** This is the most critical shift. The queries that previously hit Level 4 (generic station average) are now intercepted by Level 3. The engine isolates the localized historical behavior of that station specifically during adverse weather.
* **Level 4 (stop):** Everything fall under Level 3.
* **Level 5 (network-wide):** A tiny fraction of connections lacked even station-level data, safely defaulting to the overall network distribution to prevent algorithmic crashes.

**Conclusion:** By accounting for weather conditions, our routing engine identifies that rain increases the risk of delays. To ensure the user arrives on time despite these challenging conditions, the system automatically adapts: it selects safer transfer routes or suggests an earlier departure time. Consequently, even during severe weather, we guarantee that the user reaches their destination with the level of reliability they requested.

## 6. Interactive Demonstration Dashboard
The tool below integrates our robust routing engine into an interactive Mapbox UI. 

**Instructions for the Demo:**
1. Select a departure and destination stop.
2. Set the desired arrival deadline and confidence threshold ($Q\%$).
3. Toggle the *Weather constraint* to observe how the algorithm reroutes or suggests earlier departures to absorb meteorological risks.

In [12]:
# ==============================================================================
# INTERACTIVE DEMONSTRATION DASHBOARD
# ==============================================================================

# Launch the interactive UI using the advanced weather-aware planner.
# The dashboard automatically detects the model's capabilities and enables
# the 'Weather' dropdown toggle if the passed planner supports it.
show_dashboard(planner_wx)

interactive(children=(Dropdown(description='Start:', index=337, options=(('Bel-Air LEB', 8501170), ('Bussigny'…

# 7. Validation

To validate the robustness of our routing engine, we conducted a comprehensive backtesting analysis using historical SBB data. We selected representative origin–destination pairs across the Lausanne region, spanning various times of day. Because the planner can take a long time to run, only three pairs are displayed here; however, additional pairs were tested during development.

For each pair, we executed the routing algorithm with a random arrival deadline and a fixed confidence threshold, recording the suggested departure time and route. We then compared these suggestions against the actual historical outcomes, measuring the percentage of cases in which the user would have arrived on time based on real-world data. The results showed that our engine successfully met the specified confidence level in all cases, demonstrating its effectiveness in providing reliable journey plans even under adverse conditions.

This validation confirms that our hierarchical delay model and multi-criteria optimization approach effectively mitigate the risks of missed connections, ensuring that users can trust the suggested itineraries for their travel needs.

The major limitation of this validation model is that not all trip found in the timetable can be matched to a trip in the Istdaten database. For example, a trip marked as operating Monday–Friday in the timetable may only appear on Saturday in Istdaten. This discrepancy reduces the quality and reliability of the validation process.


In [13]:
import validate as vd

In [14]:
# Generate smaller istdaten data base

stop_list_id = stops_df["stop_id"].to_list()
istdaten_base_df = spark.table("iceberg.sbb.istdaten").filter(F.year("operating_day") >= 2024).filter(
    F.col("bpuic").isin(*stop_list_id)).filter(F.col("transit") == False)

istdaten_path = f"{hadoopfs}/user/{username}/final-project/istdaten"
istdaten_base_df.write.mode("overwrite").parquet(istdaten_path)

In [15]:
istdaten_df = spark.read.parquet(istdaten_path)

In [ ]:
random.seed(45)

for i in range(3):
    start_id, end_id = random.sample(stop_list_id, 2)
    start_name = stops_df.loc[stops_df["stop_id"] == start_id, "stop_name"].iloc[0]
    end_name = stops_df.loc[stops_df["stop_id"] == end_id, "stop_name"].iloc[0]

    hour = random.randrange(24)
    minute = random.randrange(60)
    day = random.choice(DAY_ORDER)
    
    print(f"\n\nFrom {start_name} ({start_id}) to {end_name} ({end_id}), on {day}\n \
    Arrival before {hour}h{minute}min")
    
    routes_wx = planner_std.route(
        start_id=start_id,
        end_id=end_id,
        arrival_time=f"{hour}:{minute}:00",
        day=day,
        min_confidence=0.90,
        weather="normal", # <--- Contextual weather injection
        max_routes=5,
    )

    for route in routes_wx:
        start_time = route["departure_time"]
        if start_time < 0:
            start_time += 3600*24
        print(f"Start travel at {start_time//3600}h{(start_time % 3600) // 60}min")
        on_time, late = vd.valide_route(route, istdaten_df, planner_std._walk_cache[500], debug=False)
        if late != 0:
            print(f"Miss connection at {late} time(s)")
        else:
            print("Arrive in time all the time")



From Ecublens VD, Montaney (8594890) to Ecublens VD, Ormet (8591952), on monday
     Arrival before 15h16min
Start travel at 15h0min


Arrive in time all the time
Start travel at 14h34min


Arrive in time all the time
Start travel at 14h21min


Arrive in time all the time
Start travel at 14h21min


Arrive in time all the time
Start travel at 14h21min


...

Arrive in time all the time
Start travel at 23h16min


Arrive in time all the time
Start travel at 23h12min


Arrive in time all the time


From Lausanne, Délices (8592028) to Lausanne, Benj. Constant (8591991), on monday
     Arrival before 9h51min
Start travel at 9h31min
Arrive in time all the time
Start travel at 9h30min
Arrive in time all the time
Start travel at 9h27min


Arrive in time all the time
Start travel at 9h24min


Arrive in time all the time
Start travel at 9h24min
Arrive in time all the time


We can observe that for those 3 journeys, users would always have arrive on time.

# 7.b Rain analysis

We analyzed the correlation between daily rainfall and train delays using several delay metrics (mean, median, and 90th percentile). All correlation coefficients were close to zero and statistically non-significant, indicating no meaningful relationship between rainfall and delays in our dataset. Using the selected metric, delay_median, the Spearman correlation coefficient was −0.05 (p = 0.3916), suggesting that rainfall does not provide significant predictive value for the current delay model.

However, the number of days with more than 5 mm of rainfall is relatively low, making the results difficult to interpret.

In [18]:
from validate_rain import analyze_rain_delay

In [19]:
res_rain = analyze_rain_delay(spark, hadoopfs, username)

Saved rainfall-delay histogram to: figs/rain_vs_delay_no_correlation.png

=== Daily correlations ===
delay_metric  n_days  pearson_r  pearson_p  spearman_rho  spearman_p
  delay_mean     301     -0.021     0.7144        -0.052      0.3725
delay_median     301     -0.028     0.6280        -0.050      0.3916
   delay_p90     301     -0.010     0.8651        -0.072      0.2158

Selected metric: delay_median
Spearman rho = -0.05, p = 0.3916
Interpretation: rainfall does not appear to add clear predictive value for the current delay model if the correlation is weak and non-significant.


![image](figs/rain_vs_delay_no_correlation.png)

In [21]:
# spark.stop()